# Selene ISRU Derivations

This audit notebook records the closed-form equations mirrored by the TypeScript engine and the independent Python implementation. It documents provenance, sanity checks, and known modeling gaps without changing runtime code.

Primary local references are `constants/constants.json`, `packages/engine/src/modules/*`, and `python/selene_isru/modules/*`. Literature provenance is carried in the constants `source` strings and the module notes below.


## Audit Summary

| Module | Equation family | Local implementation | Phase 0 status |
|---|---|---|---|
| Excavation | Terzaghi-McKyes cutting force plus empirical mining SEC | `excavation.ts` / `excavation.py` | Matches v1 spec. Empirical `eMining` dominates production energy; Terzaghi term is diagnostic. |
| Electrolysis | Faraday SEC, aggregate O2 yield, melt heat, VFT viscosity, limiting current | `electrolysis.ts` / `electrolysis.py` | Matches v1 spec. Aggregate yield is the deliberate Phase 1 replacement target. |
| Thermal | Constant-Cp polar sublimation, Knudsen diffusion, conductivity diagnostic | `thermal.ts` / `thermal.py` | Matches v1 spec. The 10.7 kWh/kg anchor is at `chiIce=0.005`, not default `0.05`. |
| Sabatier | Water electrolysis stoichiometry, methane conversion, van't Hoff Kp diagnostic | `sabatier.ts` / `sabatier.py` | Matches v1 spec. Optional polar-only branch. |
| Cryo | Spherical reserve tank, radiative environment, Lockheed-style MLI correlation | `cryo.ts` / `cryo.py` | Matches code/spec except the known `Nlaydens / 10` reconciliation already logged in notes. |
| Power | Solar plus RFC sizing, FSP thermal/radiator sizing, static/dynamic Pcrit, Gaussian beam capture | `power.ts` / `power.py` | Matches v1 spec. `beta-le-alpha` warning is only reachable via direct module tests. |
| Logistics | Tsiolkovsky payload estimate, packing, mission count, leverage, payback | `logistics.ts` / `logistics.py` | Matches v1 spec. Payback is a mass/productivity sizing proxy, not a financial model. |
| Construction | Hydrostatic shield balance, thermal stress, plume shear, pad yield | `construction.ts` / `construction.py` | Matches v1 spec. `pad-shear` warning is only reachable via direct module tests. |


## Excavation

The diagnostic cutting model follows the Terzaghi-McKyes style soil cutting form used for bucket/blade sizing:

```text
q  = rho_reg * g_L * z_depth
Fc = (c*Nc + q*Nq + 0.5*rho_reg*g_L*w_blade*d_blade*Ngamma) * w_blade * d_blade
Pmech = Fc * v_cut / eta_drive
```

The energy line in the Sankey is not the small blade-only mechanical term. It is the empirical fleet-level mining energy:

```text
sec_excavation = e_mining * regolith_per_kg_product
regolith_per_kg_product = 1 / (xO2 * f_extract)      # equatorial v1 aggregate
regolith_per_kg_product = 1 / chi_ice                 # polar water case
fleet_mass = k_exc_fleet * target_kg_per_day
```

Provenance: the soil-cutting factors are cited in `constants/constants.json` as Terzaghi-McKyes approximations for lunar regolith mechanics; the fleet-level mining energy is cited as a RASSOR-class engineering estimate. Phase 1 will change the equatorial throughput source from aggregate `xO2*fExtract` to the per-oxide yield, but the excavation equations themselves do not need to change.


## Molten Regolith Electrolysis

V1 used an aggregate oxygen-yield model while preserving Faraday accounting for electrical energy:

```text
sec_elec = Vcell * 4 * F / (M_O2 * eta_current)
Rreg = 1 / (xO2 * fExtract)
Qmelt = cp_reg_melt * (T_melt - T_ambient) + dH_fus
sec_thermal = Rreg * Qmelt
sec_parasitic = f_parasitic * (sec_elec + sec_thermal)
current = mdot_O2 * 4 * F / (M_O2 * eta_current)
```

Diagnostics and warnings:

```text
mu = A_mu * T_melt * exp(B_mu / (T_melt - T0_vft))
v_drain = rho_slag * g_L * h_melt^2 * sin(theta_drain) / (3 * mu)
j_limit = 4 * F * D_ox * C_bulk / delta_diff
alarm if j_operating > 0.85 * j_limit
```

Phase 1 adds the V2 composition path while keeping `oxideModel=false` as an exact aggregate fallback:

```text
o2_per_kg_i = (oxygen_atoms_i / 2) * M_O2 / M_oxide_i
dG_i(T) = A_i + B_i*T                         # per mole O2 basis
E_i(T) = -dG_i(T) / (4*F)
available_voltage = Vcell * etaCurrent
decomposed_i = E_i(Tmelt) <= available_voltage
xO2_effective = sum(mass_frac_i * o2_per_kg_i * decomposed_i) * fExtract * oxideRecoveryCalibration
Rreg = 1 / xO2_effective
```

Mass fractions are normalized only if the editable oxide vector sums above 1; sums below 1 leave the remainder as inert material. `oxideRecoveryCalibration = 1.0610956095280686` maps the representative mare vector from the V2 spec (`SiO2=0.45, TiO2=0.04, Al2O3=0.13, FeO=0.18, MgO=0.09, CaO=0.11`) and `fExtract=0.5` to the v1 continuity anchor `xO2_effective = 0.225 kg O2/kg regolith`. The Ellingham constants live in `constants/constants.json`; they are linearized per mole O2 and ordered so FeO/TiO2 decompose before the more stable oxides as voltage/current efficiency fall.

Provenance: Faraday's law and current-density limiting forms are standard electrochemical sizing relationships, with the implementation following Sibille-style molten regolith electrolysis sizing. The oxide composition vector represents mare basalt; Ellingham coefficients are JANAF/Ellingham-style linear fits with the continuity calibration above documented as a model assumption.


## Polar Sublimation And Thermal Diagnostics

The water case uses a constant-Cp closed form for cold regolith heating plus ice sublimation:

```text
sec_sub = (1 / chi_ice) * cp_reg_cold * (T_sub - T_psr) + dH_sub_ice
```

The two diagnostics are always computed:

```text
k_eff = k_c + k_r * T^3
D_K = (2/3) * r_pore * sqrt(8 * R * T / (pi * M_H2O))
```

Provenance: sublimation enthalpy and cold-regolith heat capacity are engineering reference values in `constants/constants.json`; Knudsen diffusion uses the standard pore-radius kinetic form. The v1 headline `10.7 kWh/kg H2O` corresponds to `chiIce=0.005`, while the default `chiIce=0.05` gives about `1.78 kWh/kg H2O`. This is documented in the V1 spec and retained as an explicit regression anchor.


## Sabatier Loop

The optional polar Sabatier branch keeps stoichiometry exact from molar masses:

```text
sec_water_electrolysis = V_el * 2 * F / (M_H2O * eta_faraday_el)
H2 kg/day = water kg/day * M_H2 / M_H2O
O2 kg/day = water kg/day * (M_O2 / 2) / M_H2O
CH4 kg/day = H2 kg/day * f_conversion * M_CH4 / (4 * M_H2)
q_sabatier = molar CH4 rate * abs(dH_sabatier)
Kp(T) = exp(-(dH_sabatier - T*dS_sabatier) / (R*T))
```

Provenance: water electrolysis stoichiometry and the Sabatier reaction thermodynamic diagnostic are carried by the physical constants in `constants/constants.json`. This remains a sizing and diagnostic branch, not an equilibrium reactor solve.


## Cryogenic Storage

The tank is approximated as a sphere sized from the reserve inventory:

```text
V_tank = reserve_days * product_kg_per_day / rho_cryo
r_tank = (3 * V_tank / (4*pi))^(1/3)
A_tank = 4*pi*r_tank^2
A_proj = pi*r_tank^2
```

The environment is collapsed to an equivalent hot-side sink temperature before the MLI correlation:

```text
q_env = q_solar + q_albedo + q_ir - q_space
T_hot = max((q_env / (eps_tank*sigma*A_tank))^0.25, T_tank)
T_cold = T_tank
T_m = (T_hot + T_cold) / 2
mli_flux = C1 * (N_laydens/10)^r * T_m * (T_hot - T_cold) / N_mli
         + C2 * eps_layer * (T_hot^4.67 - T_cold^4.67) / N_mli
q_leak = max(0, mli_flux * A_tank) + q_strut
boiloff = q_leak / dHvap_LOX * 86400
P_cryo = q_leak * (T_hot - T_cold) / (eta_2nd_law * T_cold)
```

Provenance: radiative terms use Stefan-Boltzmann balance; the MLI expression is the Lockheed-style correlation used by the model. The `/10` layer-density reconciliation is a documented model caveat because the stated `layers/cm` unit makes the default anchor impossible.


## Power

Solar plus RFC sizing:

```text
P_array = P_grid / eta_wire + P_grid * t_night / (t_day * eta_round_trip)
A_solar = P_array / (I_solar * eta_cell * cos(theta_sun) * F_degrade)
E_storage_Wh = P_grid * t_night / (DoD * eta_discharge)
M_storage = E_storage_Wh / SE_storage
M_solar = R_array * (P_array/1000) + M_storage
beta_solar = M_solar / (P_grid/1000)
```

Nuclear FSP sizing:

```text
eta_therm = (1 - T_sink/T_source) * eta_mech
Q_fission = P_grid / eta_therm
Q_reject = Q_fission - P_grid
A_radiator = Q_reject / (eta_rad * eps_rad * sigma * (T_sink^4 - T_env^4))
M_nuclear = M_shield + alpha_specific * (P_grid/1000)
P_crit = M_shield / (beta_solar - alpha_specific)
P_crit_dynamic = M_shield / (beta_solar/(1-d_solar)^t - alpha_specific*(1+d_nuclear*t))
```

Polar beam diagnostic:

```text
w_beam = w0_beam + theta_div_beam * z_crater_drop
eta_geo = 1 - exp(-2 * r_receiver^2 / w_beam^2)
P_floor = P_array_rim * eta_emitter * eta_geo * eta_pv_receiver
```

Provenance: solar/RFC and FSP equations are closed-form system sizing relationships from the v1 architecture spec; the Gaussian beam capture form is the v1 deliberate correction called out in the spec. The `beta-le-alpha` warning branch exists, but public bounded inputs cannot reach it after normalization.


## Logistics And Construction

Logistics uses the rocket equation to estimate delivered payload:

```text
payload = M0_LEO * exp(-dv_total / (Isp_lander * g0)) - M_dry_lander - M_resid_prop
capacity = eta_pack * payload
n_missions = ceil(total_infra_mass / capacity)
payback_days = total_infra_mass / target_kg_per_day
leverage_L = annual_product * mission_years * gear_ratio / total_infra_mass
```

Construction uses simple static balances and sizing proxies:

```text
shield_full_balance = P_internal / (rho_slag * g_L)
max_safe_cooling_delta_K = sigma_tensile * (1 - nu) / (E_slag * alpha_cte)
pad_shear = 0.5 * rho_gas_plume * v_gas_plume^2 * C_f
pad_joint_utilization = pad_shear / (tau_allowable / FS)
pad_mass = (pi/4) * d_pad^2 * t_pad * rho_slag
pads_per_year = slag_kg_per_day * 365 / pad_mass
days_to_shield_habitat = area_hab_roof * shield_design_m * rho_slag / slag_kg_per_day
```

Provenance: logistics follows Tsiolkovsky payload accounting and the construction equations are first-order structural/production balances from the v1 spec. `pad-shear` is implemented and module-tested but not reachable through public bounded params.


## V1 Regression Anchors

The authoritative executable anchors are `packages/engine/test/regression.test.ts` and `python/tests/test_regression.py`. This notebook mirrors them so future V2 changes can distinguish intentional model movement from accidental drift.


In [ ]:
from selene_isru import simulate
from selene_isru.constants import DEFAULTS
from selene_isru.modules.construction import shield_full_balance_m
from selene_isru.modules.electrolysis import melt_heat_j_per_kg, sec_elec_j_per_kg
from selene_isru.modules.logistics import payload_per_mission_kg
from selene_isru.modules.power import p_crit_kw
from selene_isru.modules.sabatier import sabatier_kp
from selene_isru.modules.thermal import sec_sub_j_per_kg

J_PER_KWH = 3_600_000
result = simulate({})

anchors = [
    ("SEC_elec MRE", sec_elec_j_per_kg(4.2, 0.9) / J_PER_KWH, "kWh/kg O2", "15.63 +/-0.5%"),
    ("xO2 effective", result["electrolysis"]["xO2Effective"], "kg/kg", "0.225 continuity"),
    ("Cp(T) melt heat", melt_heat_j_per_kg(DEFAULTS), "J/kg", "2.099805e6"),
    ("Total SEC equatorial", result["energy"]["secTotal_kWhPerKg"], "kWh/kg", "about 24.7 +/-3%"),
    ("Grid power @ 1 t/day", result["energy"]["gridPowerW"] / 1000, "kW", "about 1030 +/-3%"),
    ("Static Pcrit", p_crit_kw(1500, 250, 30), "kW", "6.818 +/-0.1%"),
    ("Polar SEC_sub chi=0.005", sec_sub_j_per_kg(0.005, 800, 40, 263) / J_PER_KWH, "kWh/kg H2O", "10.7 +/-1%"),
    ("Polar SEC_sub default chi=0.05", sec_sub_j_per_kg(0.05, 800, 40, 263) / J_PER_KWH, "kWh/kg H2O", "1.78 +/-1%"),
    ("N missions", result["logistics"]["nMissions"], "count", "1 exact"),
    ("Payback", result["logistics"]["paybackDays"], "days", "55..62"),
    ("Shield full balance", shield_full_balance_m(101325, 3000), "m", "20.85 +/-0.5%"),
    ("Payload per mission", payload_per_mission_kg(DEFAULTS), "kg", "95000..107000"),
    ("Pads per year", result["construction"]["padsPerYear"], "pads/year", "1.8..2.2"),
    ("Sabatier Kp ordering", sabatier_kp(523) / sabatier_kp(723), "ratio", "> 1"),
]

for name, value, unit, anchor in anchors:
    print(f"{name:34s} {value:12.6g} {unit:14s} anchor {anchor}")


## Standard-Library Sanity Plots

The notebook avoids plotting dependencies so it can run in the lightweight Python mirror environment. The cells below print monotonic ASCII plots for the major model sensitivities. They are not CI gates; the regression suites remain authoritative.


In [ ]:
def bar(value, lo, hi, width=36):
    if hi <= lo:
        n = 0
    else:
        n = round(max(0, min(1, (value - lo) / (hi - lo))) * width)
    return "#" * n + "." * (width - n)

def plot_series(title, rows, unit):
    values = [value for _, value in rows]
    lo, hi = min(values), max(values)
    print(title)
    for label, value in rows:
        print(f"{label:>12s} |{bar(value, lo, hi)}| {value:10.4g} {unit}")
    print()

plot_series(
    "Faraday SEC vs cell voltage",
    [(f"{v:.1f} V", sec_elec_j_per_kg(v, DEFAULTS["etaCurrent"]) / J_PER_KWH) for v in [3.5, 3.8, 4.2, 4.6, 5.0]],
    "kWh/kg",
)

plot_series(
    "Polar sublimation SEC vs ice fraction",
    [(f"chi={chi:g}", sec_sub_j_per_kg(chi, 800, 40, 263) / J_PER_KWH) for chi in [0.005, 0.01, 0.03, 0.05, 0.12]],
    "kWh/kg",
)


In [ ]:
def metric_for(patch, path):
    out = simulate(patch)
    value = out
    for key in path:
        value = value[key]
    return value

plot_series(
    "Total SEC vs target production",
    [(f"{kg:g} kg/d", metric_for({"targetKgPerDay": kg}, ["energy", "secTotal_kWhPerKg"])) for kg in [100, 500, 1000, 5000, 10000]],
    "kWh/kg",
)

plot_series(
    "Power mass trade around default",
    [(f"{kg:g} kg/d", metric_for({"targetKgPerDay": kg}, ["power", "solarMassKg"]) / 1000) for kg in [100, 500, 1000, 5000, 10000]],
    "t solar",
)

plot_series(
    "Payback vs target production",
    [(f"{kg:g} kg/d", metric_for({"targetKgPerDay": kg}, ["logistics", "paybackDays"])) for kg in [100, 500, 1000, 5000, 10000]],
    "days",
)


## Phase 1 Implementation Notes

- Per-oxide composition: Phase 1 implements the V2 representative mare vector as editable params. Phase 4 should add Mare/Highlands/PSR presets as product-level patches over these params.
- Ellingham fits: Phase 1 uses a per-mole-O2 basis and `E=-dG/(4F)` for all oxides. The constants are intentionally simple linear fits because the model is a selectivity/yield sizing layer, not a full activity-corrected melt thermodynamics solve.
- Regolith Cp(T): Phase 1 replaces constant melt Cp with `Cp(T)=a+b*T+c*T^-2` and the closed-form integral `a*(Tm-Ta)+b/2*(Tm^2-Ta^2)-c*(1/Tm-1/Ta)`, scaled by `cpRegMelt/1035` to preserve the existing slider. The default integral plus `dHfus` remains `2.099805e6 J/kg`, preserving the v1 SEC anchor.
